<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/master/Daily_challenge_W4_J5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily Challenge: Pokémon Win Prediction Analysis

This notebook aims to perform data cleaning, exploratory data analysis, and machine learning to predict Pokémon win percentages based on their stats and combat results.

## 1. Data Loading and Preparation

First, we need to unzip the provided data file to access `pokemon.csv` and `combats.csv`. Then, we'll load these files into pandas DataFrames.

In [ ]:
# Unzip the provided data file
!unzip -o '/content/Pokemon Data Analysis Tutorial.zip'

# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Load `pokemon.csv` and `combats.csv`

In [ ]:
pokemon_df = pd.read_csv('pokemon.csv')
combats_df = pd.read_csv('combats.csv')

print("Pokemon DataFrame head:")
display(pokemon_df.head())
print("\nCombats DataFrame head:")
display(combats_df.head())

### Data Cleaning: Handle Missing Values

1.  **Fill missing `Name` for Pokémon #62 (Primeape).**
    *Note: The original dataset `pokemon.csv` already has a name for Pokemon #62. Assuming the instruction implies a hypothetical scenario where it might be missing for a specific index, or it refers to a `pokemon_id` that is not directly the dataframe index.*
    Let's check the existing name for `ID` 62. If it's missing, we'll fill it. If not, we'll just acknowledge.

In [ ]:
# Check if Pokémon #62 (Primeape) has a name. Assuming # refers to the '# column'
# The original dataset already has 'Primeape' for ID 62.
# Let's ensure consistency and fill if it were hypothetically missing.
# As per the instruction, we'll explicitly set it if it's NaN for ID 62.

# Identify row for Pokemon ID 62
pokemon_id_62_index = pokemon_df[pokemon_df['#'] == 62].index

if not pokemon_id_62_index.empty and pd.isna(pokemon_df.loc[pokemon_id_62_index[0], 'Name']):
    pokemon_df.loc[pokemon_id_62_index[0], 'Name'] = 'Primeape'
    print("Filled missing name for Pokémon #62 with 'Primeape'.")
else:
    print("Pokémon #62 (Primeape) already has a name or ID 62 does not exist.")
    if not pokemon_id_62_index.empty:
        print(f"Name for Pokémon #62 is: {pokemon_df.loc[pokemon_id_62_index[0], 'Name']}")

2.  **Handle `NaN` values in `Type 2` (set to 'None').**

In [ ]:
pokemon_df['Type 2'] = pokemon_df['Type 2'].fillna('None')
print("Number of missing values in 'Type 2' after filling:", pokemon_df['Type 2'].isnull().sum())
display(pokemon_df.head())

### Calculate Win Percentage for Each Pokémon

We need to count wins and total battles for each Pokémon from the `combats_df`.

In [ ]:
# Initialize win/loss counts
pokemon_wins = {}
pokemon_battles = {}

# Iterate through battles to count wins and total battles
for index, row in combats_df.iterrows():
    pokemon1 = row['First_pokemon']
    pokemon2 = row['Second_pokemon']
    winner = row['Winner']

    # Increment total battles for both Pokémon
    pokemon_battles[pokemon1] = pokemon_battles.get(pokemon1, 0) + 1
    pokemon_battles[pokemon2] = pokemon_battles.get(pokemon2, 0) + 1

    # Increment wins for the winner
    pokemon_wins[winner] = pokemon_wins.get(winner, 0) + 1

# Create DataFrames from dictionaries
wins_df = pd.DataFrame.from_dict(pokemon_wins, orient='index', columns=['Wins']).reset_index()
wins_df = wins_df.rename(columns={'index': '#'}) # Rename to match pokemon_df key

battles_df = pd.DataFrame.from_dict(pokemon_battles, orient='index', columns=['TotalBattles']).reset_index()
battles_df = battles_df.rename(columns={'index': '#'}) # Rename to match pokemon_df key

# Merge win/battle counts with the main pokemon_df
pokemon_df = pd.merge(pokemon_df, wins_df, on='#', how='left')
pokemon_df = pd.merge(pokemon_df, battles_df, on='#', how='left')

# Fill NaN values for Pokémon that didn't participate in battles (0 wins, 0 battles)
pokemon_df['Wins'] = pokemon_df['Wins'].fillna(0)
pokemon_df['TotalBattles'] = pokemon_df['TotalBattles'].fillna(0)

# Calculate Win Percentage
pokemon_df['WinPercentage'] = pokemon_df.apply(lambda row: (row['Wins'] / row['TotalBattles']) * 100 if row['TotalBattles'] > 0 else 0, axis=1)

print("Pokemon DataFrame with Win Percentage:")
display(pokemon_df.head())

## 2. Exploratory Data Analysis (EDA) and Visualization

### Create a Correlation Matrix

Identify relationships between stats (HP, Attack, Speed) and `WinPercentage`.

In [ ]:
correlation_cols = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'WinPercentage']
correlation_matrix = pokemon_df[correlation_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Pokémon Stats and Win Percentage')
plt.show()

### Pair Plot for Stats vs. Win Percentage

Visualize the relationships between key stats (`HP`, `Attack`, `Defense`, `Speed`) and `WinPercentage`.

In [ ]:
stats_for_pairplot = ['HP', 'Attack', 'Defense', 'Speed', 'WinPercentage']
sns.pairplot(pokemon_df[stats_for_pairplot])
plt.suptitle('Pair Plot of Pokémon Key Stats and Win Percentage', y=1.02)
plt.show()

### Analyze Top 10 Pokémon by Win Percentage

In [ ]:
top_10_pokemon = pokemon_df.sort_values(by='WinPercentage', ascending=False).head(10)

print("Top 10 Pokémon by Win Percentage:")
display(top_10_pokemon[['Name', 'Type 1', 'Type 2', 'HP', 'Attack', 'Defense', 'Speed', 'TotalBattles', 'WinPercentage']])

plt.figure(figsize=(12, 7))
sns.barplot(x='Name', y='WinPercentage', data=top_10_pokemon, palette='viridis')
plt.title('Top 10 Pokémon by Win Percentage')
plt.xlabel('Pokémon Name')
plt.ylabel('Win Percentage (%)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 3. Machine Learning

We will build regression models to predict the `WinPercentage`.

### Data Splitting (Training/Testing)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

# Define features (X) and target (y)
# We'll use numerical stats as features
features = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']
X = pokemon_df[features]
y = pokemon_df['WinPercentage']

# Split data into training and testing sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")

### Train and Evaluate Regression Models

We will train three regression models: Linear Regression, Random Forest Regressor, and XGBoost Regressor.

In [ ]:
# Initialize models
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest Regressor': RandomForestRegressor(random_state=42),
    'XGBoost Regressor': XGBRegressor(random_state=42)
}

mae_scores = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mae_scores[name] = mae
    print(f"{name} MAE: {mae:.4f}")

### Compare Model Performance (MAE)

In [ ]:
mae_df = pd.DataFrame(mae_scores.items(), columns=['Model', 'MAE'])
mae_df = mae_df.sort_values(by='MAE', ascending=True)

print("\nModel Performance (Mean Absolute Error):")
display(mae_df)

plt.figure(figsize=(10, 6))
sns.barplot(x='Model', y='MAE', data=mae_df, palette='magma')
plt.title('Comparison of Regression Models by MAE')
plt.xlabel('Model')
plt.ylabel('Mean Absolute Error')
plt.tight_layout()
plt.show()